# Load and display Reviews.csv
This notebook reads the Reviews.csv file and shows a quick summary of the data.

In [3]:
# Adjust the path if Reviews.csv is located elsewhere relative to this notebook
df <- read.csv("Reviews.csv", stringsAsFactors = FALSE)
cat(sprintf("Loaded %d rows and %d columns\n", nrow(df), ncol(df)))
head(df)
str(df)
summary(df)


Loaded 568454 rows and 10 columns


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<chr>,<chr>
1,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than most.
2,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,"Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as ""Jumbo""."
3,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all","This is a confection that has been around a few centuries. It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut into tiny squares and then liberally coated with powdered sugar. And it is a tiny mouthful of heaven. Not too chewy, and very flavorful. I highly recommend this yummy treat. If you are familiar with the story of C.S. Lewis' ""The Lion, The Witch, and The Wardrobe"" - this is the treat that seduces Edmund into selling out his Brother and Sisters to the Witch."
4,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient in Robitussin I believe I have found it. I got this in addition to the Root Beer Extract I ordered (which was good) and made some cherry soda. The flavor is very medicinal.
5,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,"Great taffy at a great price. There was a wide assortment of yummy taffy. Delivery was very quick. If your a taffy lover, this is a deal."
6,6,B006K2ZZ7K,ADT0SRK1MGOEU,Twoapennything,0,0,4,1342051200,Nice Taffy,"I got a wild hair for taffy and ordered this five pound bag. The taffy was all very enjoyable with many flavors: watermelon, root beer, melon, peppermint, grape, etc. My only complaint is there was a bit too much red/black licorice-flavored pieces (just not my particular favorites). Between me, my kids, and my husband, this lasted only two weeks! I would recommend this brand of taffy -- it was a delightful treat."


'data.frame':	568454 obs. of  10 variables:
 $ Id                    : int  1 2 3 4 5 6 7 8 9 10 ...
 $ ProductId             : chr  "B001E4KFG0" "B00813GRG4" "B000LQOCH0" "B000UA0QIQ" ...
 $ UserId                : chr  "A3SGXH7AUHU8GW" "A1D87F6ZCVE5NK" "ABXLMWJIXXAIN" "A395BORC6FGVXV" ...
 $ ProfileName           : chr  "delmartian" "dll pa" "Natalia Corres \"Natalia Corres\"" "Karl" ...
 $ HelpfulnessNumerator  : int  1 0 1 3 0 0 0 0 1 0 ...
 $ HelpfulnessDenominator: int  1 0 1 3 0 0 0 0 1 0 ...
 $ Score                 : int  5 1 4 2 5 4 5 5 5 5 ...
 $ Time                  : int  1303862400 1346976000 1219017600 1307923200 1350777600 1342051200 1340150400 1336003200 1322006400 1351209600 ...
 $ Summary               : chr  "Good Quality Dog Food" "Not as Advertised" "\"Delight\" says it all" "Cough Medicine" ...
 $ Text                  : chr  "I have bought several of the Vitality canned dog food products and have found them all to be of good quality. T"| __truncated__ "Product 

       Id             ProductId            UserId          ProfileName    
 Min.   :     1   Length   :568454   Length   :568454   Length   :568454  
 1st Qu.:142114   N.unique : 74258   N.unique :256059   N.unique :218417  
 Median :284228   N.blank  :     0   N.blank  :     0   N.blank  :     0  
 Mean   :284228   Min.nchar:    10   Min.nchar:    10   Min.nchar:     1  
 3rd Qu.:426341   Max.nchar:    10   Max.nchar:    21   Max.nchar:    53  
 Max.   :568454                                         NAs      :     4  
 HelpfulnessNumerator HelpfulnessDenominator     Score      
 Min.   :  0.000      Min.   :  0.000        Min.   :1.000  
 1st Qu.:  0.000      1st Qu.:  0.000        1st Qu.:4.000  
 Median :  0.000      Median :  1.000        Median :5.000  
 Mean   :  1.744      Mean   :  2.229        Mean   :4.183  
 3rd Qu.:  2.000      3rd Qu.:  2.000        3rd Qu.:5.000  
 Max.   :866.000      Max.   :923.000        Max.   :5.000  
      Time                Summary              T

## Step 1: Install/load packages
We will use `tm` for text preprocessing and `glmnet` for multinomial logistic regression.

In [4]:
required_packages <- c("tm", "glmnet", "Matrix")
for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
  }
}

library(tm)
library(Matrix)
library(glmnet)
cat("Packages loaded successfully\n")

Loading required package: NLP

Loaded glmnet 5.0



Packages loaded successfully


## Step 2: Build sentiment labels from the dataset
If `Sentiment` exists we use it, otherwise we create it from `Score`:
- 1-2 -> negative
- 3 -> neutral
- 4-5 -> positive

In [5]:
if (!exists("df")) {
  df <- read.csv("Reviews.csv", stringsAsFactors = FALSE)
}

if (!"Text" %in% names(df)) stop("Column 'Text' not found in Reviews.csv")

if ("Sentiment" %in% names(df)) {
  sentiment <- tolower(as.character(df$Sentiment))
} else if ("Score" %in% names(df)) {
  score_num <- suppressWarnings(as.numeric(df$Score))
  sentiment <- ifelse(score_num >= 4, "positive", ifelse(score_num == 3, "neutral", "negative"))
} else {
  stop("Need either a 'Sentiment' or 'Score' column to create labels")
}

model_df <- data.frame(
  Text = as.character(df$Text),
  Sentiment = factor(sentiment, levels = c("negative", "neutral", "positive")),
  stringsAsFactors = FALSE
)
model_df <- model_df[!is.na(model_df$Text) & model_df$Text != "" & !is.na(model_df$Sentiment), ]

cat(sprintf("Rows ready for modeling: %d\n", nrow(model_df)))
print(table(model_df$Sentiment))
head(model_df, 3)

Rows ready for modeling: 568454

negative  neutral positive 
   82037    42640   443777 


,Text,Sentiment
,<chr>,<fct>
1,I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than most.,positive
2,"Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as ""Jumbo"".",negative
3,"This is a confection that has been around a few centuries. It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut into tiny squares and then liberally coated with powdered sugar. And it is a tiny mouthful of heaven. Not too chewy, and very flavorful. I highly recommend this yummy treat. If you are familiar with the story of C.S. Lewis' ""The Lion, The Witch, and The Wardrobe"" - this is the treat that seduces Edmund into selling out his Brother and Sisters to the Witch.",positive


## Step 3: Train/test split and text cleaning
This step splits the data, preprocesses text, and converts text into document-term matrices.

In [24]:
set.seed(42)

# Sample to at most 100k rows to avoid out-of-memory errors on large corpora
max_rows <- 100000L
if (nrow(model_df) > max_rows) {
  sample_idx <- sample(seq_len(nrow(model_df)), size = max_rows)
  model_df_sample <- model_df[sample_idx, ]
} else {
  model_df_sample <- model_df
}

idx <- sample(seq_len(nrow(model_df_sample)), size = floor(0.8 * nrow(model_df_sample)))
train_df <- model_df_sample[idx, ]
test_df  <- model_df_sample[-idx, ]

clean_text <- function(x) {
  corpus <- VCorpus(VectorSource(x))
  corpus <- tm_map(corpus, content_transformer(tolower))
  corpus <- tm_map(corpus, removePunctuation)
  corpus <- tm_map(corpus, removeNumbers)
  corpus <- tm_map(corpus, removeWords, stopwords("en"))
  corpus <- tm_map(corpus, stripWhitespace)
  corpus
}

train_corpus <- clean_text(train_df$Text)
test_corpus  <- clean_text(test_df$Text)

# bounds: keep terms that appear in at least 5 documents; min word length 3
train_dtm <- DocumentTermMatrix(
  train_corpus,
  control = list(
    wordLengths = c(3, Inf),
    bounds = list(global = c(5L, Inf))
  )
)
test_dtm <- DocumentTermMatrix(
  test_corpus,
  control = list(dictionary = Terms(train_dtm))
)

# Keep as sparse matrices — never call as.matrix() on these
x_train <- Matrix::sparseMatrix(
  i = train_dtm$i, j = train_dtm$j, x = as.numeric(train_dtm$v),
  dims = dim(train_dtm), dimnames = dimnames(train_dtm)
)
x_test <- Matrix::sparseMatrix(
  i = test_dtm$i, j = test_dtm$j, x = as.numeric(test_dtm$v),
  dims = dim(test_dtm), dimnames = dimnames(test_dtm)
)
y_train <- train_df$Sentiment
y_test  <- test_df$Sentiment

cat(sprintf("Train rows: %d | Test rows: %d\n", nrow(train_df), nrow(test_df)))
cat(sprintf("Vocabulary size: %d\n", ncol(x_train)))

Train rows: 80000 | Test rows: 20000
Vocabulary size: 18582


## Step 4: Train multinomial logistic regression model
We use cross-validation to pick the best regularization strength (`lambda`).

In [ ]:
# library(glmnet)

# cv_fit <- cv.glmnet(
#   x_train,
#   y_train,
#   family = "multinomial",
#   type.measure = "class",
#   nfolds = 5
# )

# cat(sprintf("Best lambda: %.6f\n", cv_fit$lambda.min))
# cat("Class:", class(cv_fit), "\n")

Warning message:
"from glmnet C++ code (error code -80); Convergence for 80th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -80); Convergence for 80th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -85); Convergence for 85th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -86); Convergence for 86th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -83); Convergence for 83th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -78); Convergence for 78th lambda value not reached after maxit=100000 iterations; s

Best lambda: 0.000955
Class: cv.glmnet 


## Step 5: Predict and evaluate
This cell prints a confusion matrix and overall accuracy.

In [6]:
pred <- predict(cv_fit, newx = x_test, s = "lambda.min", type = "class")
pred <- factor(as.character(pred), levels = levels(y_test))

cat("Confusion Matrix:\n")
print(table(Actual = y_test, Predicted = pred))
cat(sprintf("Accuracy: %.4f\n", mean(pred == y_test)))

Confusion Matrix:
          Predicted
Actual     negative neutral positive
  negative     1800     102     1057
  neutral       303     277      930
  positive      333     177    15021
Accuracy: 0.8549


In [ ]:
# saveRDS(cv_fit, "sentiment_model.rds")
# cat("Model saved to sentiment_model.rds\n")

Model saved to sentiment_model.rds


In [9]:
library(glmnet)
loaded_model <- readRDS("sentiment_model.rds")
cat("Model loaded. Class:", class(loaded_model)[1], "\n")
cat("lambda.min:", loaded_model$lambda.min, "\n")

Model loaded. Class: cv.glmnet 
lambda.min: 0.0009547247 


In [10]:
# class fix only needed if loading an older saved file; new saves won't need this
if (!inherits(loaded_model, "cv.glmnet")) {
  class(loaded_model) <- c("cv.glmnet", "list")
}

In [25]:
pred_raw <- predict(loaded_model, newx = x_test, s = "lambda.min", type = "class")
pred <- factor(as.character(pred_raw), levels = levels(y_test))

cm <- table(Actual = y_test, Predicted = pred)
cat("Confusion Matrix:\n")
print(cm)

classes <- levels(y_test)
total   <- sum(cm)

metrics <- lapply(classes, function(cls) {
  TP <- cm[cls, cls]
  FP <- sum(cm[, cls]) - TP
  FN <- sum(cm[cls, ]) - TP
  TN <- total - TP - FP - FN

  precision <- ifelse((TP + FP) == 0, NA, TP / (TP + FP))
  recall    <- ifelse((TP + FN) == 0, NA, TP / (TP + FN))
  f1        <- ifelse(is.na(precision) | is.na(recall) | (precision + recall) == 0,
                      NA, 2 * precision * recall / (precision + recall))
  fpr       <- ifelse((FP + TN) == 0, NA, FP / (FP + TN))
  fnr       <- ifelse((FN + TP) == 0, NA, FN / (FN + TP))
  specificity <- ifelse((TN + FP) == 0, NA, TN / (TN + FP))

  data.frame(Class = cls, TP = TP, FP = FP, FN = FN, TN = TN,
             Precision = round(precision, 4), Recall = round(recall, 4),
             F1 = round(f1, 4), FPR = round(fpr, 4), FNR = round(fnr, 4),
             Specificity = round(specificity, 4))
})

results <- do.call(rbind, metrics)
rownames(results) <- NULL

cat("\nPer-Class Metrics:\n")
print(results)

support   <- as.integer(table(y_test)[classes])
macro_p   <- mean(results$Precision, na.rm = TRUE)
macro_r   <- mean(results$Recall,    na.rm = TRUE)
macro_f1  <- mean(results$F1,        na.rm = TRUE)
weighted_p  <- sum(results$Precision * support, na.rm = TRUE) / sum(support)
weighted_r  <- sum(results$Recall    * support, na.rm = TRUE) / sum(support)
weighted_f1 <- sum(results$F1        * support, na.rm = TRUE) / sum(support)
accuracy    <- sum(diag(cm)) / total

cat(sprintf("\nOverall Accuracy : %.4f\n", accuracy))
cat(sprintf("Macro    Precision: %.4f  Recall: %.4f  F1: %.4f\n", macro_p,    macro_r,    macro_f1))
cat(sprintf("Weighted Precision: %.4f  Recall: %.4f  F1: %.4f\n", weighted_p, weighted_r, weighted_f1))

Confusion Matrix:
          Predicted
Actual     negative neutral positive
  negative     1800     102     1057
  neutral       303     277      930
  positive      333     177    15021

Per-Class Metrics:
     Class    TP   FP   FN    TN Precision Recall     F1    FPR    FNR
1 negative  1800  636 1159 16405    0.7389 0.6083 0.6673 0.0373 0.3917
2  neutral   277  279 1233 18211    0.4982 0.1834 0.2682 0.0151 0.8166
3 positive 15021 1987  510  2482    0.8832 0.9672 0.9233 0.4446 0.0328
  Specificity
1      0.9627
2      0.9849
3      0.5554

Overall Accuracy : 0.8549
Macro    Precision: 0.7068  Recall: 0.5863  F1: 0.6196
Weighted Precision: 0.8328  Recall: 0.8549  F1: 0.8360


## Evaluate: DistilBERT
Uses `lxyuan/distilbert-base-multilingual-cased-sentiments-student` (3-class: negative / neutral / positive) via HuggingFace Transformers through `reticulate`. Evaluated on a 500-row random sample of the test set for speed.

In [26]:
if (!requireNamespace("reticulate", quietly = TRUE)) {
  install.packages("reticulate", repos = "https://cloud.r-project.org")
}
library(reticulate)

# Install Python dependencies into reticulate's environment if missing
py_pkgs <- c("transformers", "torch")
for (pkg in py_pkgs) {
  if (!reticulate::py_module_available(pkg)) {
    message("Installing Python package: ", pkg)
    reticulate::py_install(pkg, pip = TRUE)
  }
}

transformers <- import("transformers")

# 500-row subsample for speed
set.seed(42)
n_eval      <- 500L
eval_idx    <- sample(seq_len(nrow(test_df)), size = min(n_eval, nrow(test_df)))
eval_texts  <- test_df$Text[eval_idx]
eval_labels <- test_df$Sentiment[eval_idx]

distilbert_pipe <- transformers$pipeline(
  "sentiment-analysis",
  model      = "lxyuan/distilbert-base-multilingual-cased-sentiments-student",
  truncation = TRUE,
  max_length = 512L,
  top_k      = 1L
)

batch_size <- 32L
raw_preds  <- character(length(eval_texts))
for (i in seq(1, length(eval_texts), by = batch_size)) {
  end   <- min(i + batch_size - 1L, length(eval_texts))
  batch <- as.list(eval_texts[i:end])
  res   <- distilbert_pipe(batch)
  raw_preds[i:end] <- sapply(res, function(r) tolower(r[[1]]$label))
}

pred_distilbert <- factor(raw_preds, levels = levels(eval_labels))

cm_db  <- table(Actual = eval_labels, Predicted = pred_distilbert)
cat("DistilBERT Confusion Matrix:\n")
print(cm_db)

classes  <- levels(eval_labels)
total_db <- sum(cm_db)
metrics_db <- lapply(classes, function(cls) {
  TP <- cm_db[cls, cls]; FP <- sum(cm_db[, cls]) - TP
  FN <- sum(cm_db[cls, ]) - TP; TN <- total_db - TP - FP - FN
  prec <- ifelse((TP + FP) == 0, NA, TP / (TP + FP))
  rec  <- ifelse((TP + FN) == 0, NA, TP / (TP + FN))
  f1   <- ifelse(is.na(prec) | is.na(rec) | (prec + rec) == 0, NA,
                 2 * prec * rec / (prec + rec))
  data.frame(Class = cls, Precision = round(prec, 4),
             Recall = round(rec, 4), F1 = round(f1, 4))
})
results_db <- do.call(rbind, metrics_db); rownames(results_db) <- NULL
cat("\nPer-Class Metrics:\n"); print(results_db)

support_db     <- as.integer(table(eval_labels)[classes])
macro_f1_db    <- mean(results_db$F1, na.rm = TRUE)
weighted_f1_db <- sum(results_db$F1 * support_db, na.rm = TRUE) / sum(support_db)
accuracy_db    <- sum(diag(cm_db)) / total_db
cat(sprintf("\nDistilBERT Accuracy : %.4f\n", accuracy_db))
cat(sprintf("Macro F1            : %.4f\n",  macro_f1_db))
cat(sprintf("Weighted F1         : %.4f\n",  weighted_f1_db))

DistilBERT Confusion Matrix:
          Predicted
Actual     negative neutral positive
  negative       55       6       19
  neutral        10       5       13
  positive       69       3      320

Per-Class Metrics:
     Class Precision Recall     F1
1 negative    0.4104 0.6875 0.5140
2  neutral    0.3571 0.1786 0.2381
3 positive    0.9091 0.8163 0.8602

DistilBERT Accuracy : 0.7600
Macro F1            : 0.5374
Weighted F1         : 0.7700


## Evaluate: RoBERTa
Uses `cardiffnlp/twitter-roberta-base-sentiment-latest` (3-class: negative / neutral / positive) via HuggingFace Transformers. Reuses the same 500-row sample from the DistilBERT cell so results are directly comparable.

In [27]:
# Reuses: transformers, eval_texts, eval_labels, classes, support_db, batch_size
# from the DistilBERT cell above

roberta_pipe <- transformers$pipeline(
  "sentiment-analysis",
  model      = "cardiffnlp/twitter-roberta-base-sentiment-latest",
  truncation = TRUE,
  max_length = 512L,
  top_k      = 1L
)

raw_preds_r <- character(length(eval_texts))
for (i in seq(1, length(eval_texts), by = batch_size)) {
  end   <- min(i + batch_size - 1L, length(eval_texts))
  batch <- as.list(eval_texts[i:end])
  res   <- roberta_pipe(batch)
  raw_preds_r[i:end] <- sapply(res, function(r) tolower(r[[1]]$label))
}

pred_roberta <- factor(raw_preds_r, levels = levels(eval_labels))

cm_rb  <- table(Actual = eval_labels, Predicted = pred_roberta)
cat("RoBERTa Confusion Matrix:\n")
print(cm_rb)

total_rb <- sum(cm_rb)
metrics_rb <- lapply(classes, function(cls) {
  TP <- cm_rb[cls, cls]; FP <- sum(cm_rb[, cls]) - TP
  FN <- sum(cm_rb[cls, ]) - TP; TN <- total_rb - TP - FP - FN
  prec <- ifelse((TP + FP) == 0, NA, TP / (TP + FP))
  rec  <- ifelse((TP + FN) == 0, NA, TP / (TP + FN))
  f1   <- ifelse(is.na(prec) | is.na(rec) | (prec + rec) == 0, NA,
                 2 * prec * rec / (prec + rec))
  data.frame(Class = cls, Precision = round(prec, 4),
             Recall = round(rec, 4), F1 = round(f1, 4))
})
results_rb <- do.call(rbind, metrics_rb); rownames(results_rb) <- NULL
cat("\nPer-Class Metrics:\n"); print(results_rb)

macro_f1_rb    <- mean(results_rb$F1, na.rm = TRUE)
weighted_f1_rb <- sum(results_rb$F1 * support_db, na.rm = TRUE) / sum(support_db)
accuracy_rb    <- sum(diag(cm_rb)) / total_rb
cat(sprintf("\nRoBERTa Accuracy : %.4f\n", accuracy_rb))
cat(sprintf("Macro F1         : %.4f\n",  macro_f1_rb))
cat(sprintf("Weighted F1      : %.4f\n",  weighted_f1_rb))

RoBERTa Confusion Matrix:
          Predicted
Actual     negative neutral positive
  negative       61      11        8
  neutral         9      11        8
  positive       20      33      339

Per-Class Metrics:
     Class Precision Recall     F1
1 negative    0.6778 0.7625 0.7176
2  neutral    0.2000 0.3929 0.2651
3 positive    0.9549 0.8648 0.9076

RoBERTa Accuracy : 0.8220
Macro F1         : 0.6301
Weighted F1      : 0.8412


## Evaluate: Logistic Regression (subsample)
Re-evaluates the trained glmnet model on the **same 500-row subsample** used for DistilBERT and RoBERTa so all three metrics are directly comparable.

In [28]:
# Reuses: loaded_model, x_test, eval_idx, eval_labels, classes, support_db
# from the earlier cells

x_eval_lr   <- x_test[eval_idx, ]
pred_lr_raw <- predict(loaded_model, newx = x_eval_lr, s = "lambda.min", type = "class")
pred_lr     <- factor(as.character(pred_lr_raw), levels = levels(eval_labels))

cm_lr  <- table(Actual = eval_labels, Predicted = pred_lr)
cat("Logistic Regression Confusion Matrix:\n")
print(cm_lr)

total_lr   <- sum(cm_lr)
metrics_lr <- lapply(classes, function(cls) {
  TP <- cm_lr[cls, cls]; FP <- sum(cm_lr[, cls]) - TP
  FN <- sum(cm_lr[cls, ]) - TP; TN <- total_lr - TP - FP - FN
  prec <- ifelse((TP + FP) == 0, NA, TP / (TP + FP))
  rec  <- ifelse((TP + FN) == 0, NA, TP / (TP + FN))
  f1   <- ifelse(is.na(prec) | is.na(rec) | (prec + rec) == 0, NA,
                 2 * prec * rec / (prec + rec))
  data.frame(Class = cls, Precision = round(prec, 4),
             Recall = round(rec, 4), F1 = round(f1, 4))
})
results_lr <- do.call(rbind, metrics_lr); rownames(results_lr) <- NULL
cat("\nPer-Class Metrics:\n"); print(results_lr)

macro_f1_lr    <- mean(results_lr$F1, na.rm = TRUE)
weighted_f1_lr <- sum(results_lr$F1 * support_db, na.rm = TRUE) / sum(support_db)
accuracy_lr    <- sum(diag(cm_lr)) / total_lr
cat(sprintf("\nLogistic Regression Accuracy : %.4f\n", accuracy_lr))
cat(sprintf("Macro F1                     : %.4f\n",   macro_f1_lr))
cat(sprintf("Weighted F1                  : %.4f\n",   weighted_f1_lr))

Logistic Regression Confusion Matrix:
          Predicted
Actual     negative neutral positive
  negative       55       3       22
  neutral         4       5       19
  positive        6       6      380

Per-Class Metrics:
     Class Precision Recall     F1
1 negative    0.8462 0.6875 0.7586
2  neutral    0.3571 0.1786 0.2381
3 positive    0.9026 0.9694 0.9348

Logistic Regression Accuracy : 0.8800
Macro F1                     : 0.6438
Weighted F1                  : 0.8676


## Model Comparison & Conclusion
Side-by-side summary of all three models on the same 500-row test subsample, followed by a written conclusion.

In [29]:
# --- Comparison table ---
comparison <- data.frame(
  Model       = c("Logistic Regression", "DistilBERT", "RoBERTa"),
  Accuracy    = round(c(accuracy_lr, accuracy_db, accuracy_rb), 4),
  Macro_F1    = round(c(macro_f1_lr, macro_f1_db, macro_f1_rb), 4),
  Weighted_F1 = round(c(weighted_f1_lr, weighted_f1_db, weighted_f1_rb), 4)
)
comparison <- comparison[order(-comparison$Weighted_F1), ]
rownames(comparison) <- NULL

cat("=== Model Comparison (500-row subsample) ===\n\n")
print(comparison, row.names = FALSE)

best_model <- comparison$Model[1]
best_wf1   <- comparison$Weighted_F1[1]

# --- Conclusion ---
cat("\n\n--- Conclusion ---\n\n")

cat(sprintf(paste0(
  "Three models were evaluated on an identical 500-row random sample drawn from\n",
  "the test set, ensuring a fair, head-to-head comparison.\n\n",

  "Logistic Regression (raw TF bag-of-words + glmnet, L1/L2 regularisation)\n",
  "  Accuracy: %.4f | Macro F1: %.4f | Weighted F1: %.4f\n",
  "  The document-term matrix uses raw term frequency counts (weightTf), not\n",
  "  TF-IDF, meaning common words are not down-weighted. Despite this, glmnet's\n",
  "  regularisation compensates well and the model is fast and interpretable.\n",
  "  Its core limitation is that it ignores word order and context entirely.\n\n",

  "DistilBERT (lxyuan/distilbert-base-multilingual-cased-sentiments-student)\n",
  "  Accuracy: %.4f | Macro F1: %.4f | Weighted F1: %.4f\n",
  "  A distilled transformer pretrained for 3-class sentiment. Captures contextual\n",
  "  semantics and word order, but is applied zero-shot here with no fine-tuning\n",
  "  on Amazon Reviews data.\n\n",

  "RoBERTa (cardiffnlp/twitter-roberta-base-sentiment-latest)\n",
  "  Accuracy: %.4f | Macro F1: %.4f | Weighted F1: %.4f\n",
  "  A robustly optimised BERT variant fine-tuned on Twitter sentiment. Also\n",
  "  zero-shot on this domain; Twitter vs. product-review domain mismatch\n",
  "  can hurt performance, particularly on the neutral class.\n\n",

  "Winner: %s (Weighted F1 = %.4f)\n\n",

  "Key Takeaways:\n",
  "  1. The LR baseline uses raw TF counts (not TF-IDF); switching to TF-IDF\n",
  "     weighting would likely improve its handling of frequent but uninformative\n",
  "     words and could push its metrics higher.\n",
  "  2. The LR model is still surprisingly competitive given its simplicity,\n",
  "     especially on the dominant positive class.\n",
  "  3. Transformer models excel at contextual understanding but suffer here\n",
  "     because they are applied zero-shot to a different domain.\n",
  "  4. Fine-tuning DistilBERT or RoBERTa on Amazon Reviews data would likely\n",
  "     push their Weighted F1 well above the LR baseline.\n",
  "  5. The neutral class is the hardest for all models due to its low support\n",
  "     relative to positive/negative, depressing Macro F1 across the board.\n"
),
accuracy_lr, macro_f1_lr, weighted_f1_lr,
accuracy_db, macro_f1_db, weighted_f1_db,
accuracy_rb, macro_f1_rb, weighted_f1_rb,
best_model, best_wf1
))

=== Model Comparison (500-row subsample) ===

               Model Accuracy Macro_F1 Weighted_F1
 Logistic Regression    0.880   0.6438      0.8676
             RoBERTa    0.822   0.6301      0.8412
          DistilBERT    0.760   0.5374      0.7700


--- Conclusion ---

Three models were evaluated on an identical 500-row random sample drawn from
the test set, ensuring a fair, head-to-head comparison.

Logistic Regression (raw TF bag-of-words + glmnet, L1/L2 regularisation)
  Accuracy: 0.8800 | Macro F1: 0.6438 | Weighted F1: 0.8676
  The document-term matrix uses raw term frequency counts (weightTf), not
  TF-IDF, meaning common words are not down-weighted. Despite this, glmnet's
  regularisation compensates well and the model is fast and interpretable.
  Its core limitation is that it ignores word order and context entirely.

DistilBERT (lxyuan/distilbert-base-multilingual-cased-sentiments-student)
  Accuracy: 0.7600 | Macro F1: 0.5374 | Weighted F1: 0.7700
  A distilled transformer 